# 02 - Frozen features and evaluation

Run the `proj_out` arrays from notebook 01 through BrainOmni's blocks, pool into one vector per
subject, and evaluate AD vs control.

- Pipeline: `blocks[:-1]` → L2 normalise → mean over time → average the two segments → flatten. Then logistic regression, stratified 5-fold, seed 123.
- Mirrors `00_extract_frozen_features` and `01_baselines` on the sensor cohort, so the two match in method.
- **Frozen only.** Fine-tuning cost ~40 hours on the sensor data and made results *worse*, so not worth spending before knowing whether frozen features work here at all.
- **Caveat.** 161 subjects, not the 148 from OSF, and possibly overlapping. Not directly comparable to the 0.831 sensor result. The question here is narrower: does BrainOmni produce usable representations from parcellated input with reconstructed geometry?


In [13]:
import os, sys, json
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, torch
import torch.nn.functional as F
sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/BrainOmni')
from brainomni.model import BrainOmni

OUT = '../results_parcel'
CKPT_ROOT = '/Users/faenegoro/Documents/Neuroimaging/BrainOmni/ckpt_collection'
MODEL, SUFFIX = 'tiny', 'glasser52'      # must match notebook 01

device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')

proj1  = np.load(f'{OUT}/parcel_proj_out1_{MODEL}_{SUFFIX}.npy')
proj2  = np.load(f'{OUT}/parcel_proj_out2_{MODEL}_{SUFFIX}.npy')
labels = np.load(f'{OUT}/parcel_labels_{SUFFIX}.npy')
N = len(labels)
print(f'proj_out: {proj1.shape} | {N} subjects | AD {int((labels==1).sum())} / Control {int((labels==0).sum())}')
print(f'majority-class baseline: {max((labels==1).mean(), (labels==0).mean()):.3f}')

proj_out: (161, 16, 160, 256) | 161 subjects | AD 77 / Control 84
majority-class baseline: 0.522


In [14]:
cfg  = json.load(open(f'{CKPT_ROOT}/{MODEL}/model_cfg.json'))
ckpt = torch.load(f'{CKPT_ROOT}/{MODEL}/BrainOmni.pt', map_location='cpu', weights_only=True)
model = BrainOmni(**cfg); model.load_state_dict(ckpt, strict=False)
for p in model.parameters():
    p.requires_grad = False
model = model.to(device).eval()

def extract_frozen(p1, p2):
    """proj_out -> transformer blocks -> L2 normalize -> mean over time -> average the two
    segments -> flatten. Same definition used for the sensor-level frozen features."""
    feats = []
    with torch.no_grad():
        for i in range(N):
            x1 = torch.tensor(p1[i]).unsqueeze(0).to(device)
            x2 = torch.tensor(p2[i]).unsqueeze(0).to(device)
            for b in list(model.blocks)[:-1]:
                x1 = b(x1); x2 = b(x2)
            x1 = F.normalize(x1, p=2.0, dim=-1, eps=1e-6).mean(dim=2)
            x2 = F.normalize(x2, p=2.0, dim=-1, eps=1e-6).mean(dim=2)
            feats.append(((x1 + x2) / 2).flatten().cpu().numpy())
    return np.stack(feats)

X = extract_frozen(proj1, proj2)
np.save(f'{OUT}/parcel_frozen_{MODEL}_{SUFFIX}.npy', X)
print('frozen features:', X.shape)
print('finite?', bool(np.isfinite(X).all()), '| range', float(X.min()), 'to', float(X.max()))

/opt/miniconda3/envs/brainomni/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


frozen features: (161, 4096)
finite? True | range -0.41430217027664185 to 0.23324289917945862


In [15]:
# 5-fold CV, seed 123, same protocol as the sensor-level baselines.
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(5, shuffle=True, random_state=123)
r  = cross_validate(LogisticRegression(max_iter=2000), X, labels, cv=cv,
                    scoring=['accuracy', 'roc_auc'])
acc, auc = r['test_accuracy'].mean(), r['test_roc_auc'].mean()
print(f'Frozen BrainOmni-{MODEL} on parcels ({SUFFIX}):')
print(f'  accuracy {acc:.3f}   (per fold: {np.round(r["test_accuracy"], 3)})')
print(f'  AUC      {auc:.3f}   (per fold: {np.round(r["test_roc_auc"], 3)})')

Frozen BrainOmni-tiny on parcels (glasser52):
  accuracy 0.869   (per fold: [1.    0.812 0.875 0.844 0.812])
  AUC      0.911   (per fold: [1.    0.844 0.937 0.898 0.878])


In [16]:
# Decision gate: is this working, or is something broken upstream?
chance = max((labels == 1).mean(), (labels == 0).mean())
print(f'majority-class baseline : {chance:.3f}')
print(f'this result             : {acc:.3f}  (AUC {auc:.3f})')
print()
if acc < chance + 0.05:
    print('NEAR CHANCE. Something upstream is likely broken rather than this being a real')
    print('null result. Worth checking before interpreting:')
    print('  - sign flipping: if not applied, regions are inverted in a random subset of subjects')
    print('  - geometry: re-check the centroid plot in notebook 00')
    print('  - feature range: compare against results_sensor/frozen_base.npy')
elif acc < 0.70:
    print('WEAK. Above chance but well below the sensor-level result, consistent with')
    print('source-space input being out of distribution for a sensor-pretrained model.')
else:
    print('WORKING. BrainOmni produces usable representations from parcellated input')
    print('with reconstructed geometry. Fine-tuning would now be worth considering.')
print()
print('For reference, sensor-level frozen BrainOmni-base on the OSF cohort reached 0.831 / 0.878,')
print('but that is a different cohort (148 vs 161 subjects), so treat it as context, not a')
print('like-for-like comparison.')

majority-class baseline : 0.522
this result             : 0.869  (AUC 0.911)

WORKING. BrainOmni produces usable representations from parcellated input
with reconstructed geometry. Fine-tuning would now be worth considering.

For reference, sensor-level frozen BrainOmni-base on the OSF cohort reached 0.831 / 0.878,
but that is a different cohort (148 vs 161 subjects), so treat it as context, not a
like-for-like comparison.


## Parcellation resolution: 210 regions versus 52 parcels

Runs only once both frozen feature files exist, so set `SUFFIX` to each in turn and rerun
the notebook above before reaching this cell.

The comparison is a paired bootstrap on the same 161 subjects and the same resamples, which
is the right test here because the two feature sets describe identical people. Comparing
their confidence intervals for overlap would be weaker.

Metrics are pooled out-of-fold, so they differ slightly from the per-fold means printed
above. The paired test needs pooled predictions, so quote these when citing the test.

In [17]:
from sklearn.metrics import accuracy_score, roc_auc_score

A = f'{OUT}/parcel_frozen_{MODEL}_cortical.npy'    # 210 Brainnetome regions
B = f'{OUT}/parcel_frozen_{MODEL}_glasser52.npy'   # 52 Glasser parcels

if not (os.path.exists(A) and os.path.exists(B)):
    print('need both frozen files, run this notebook once per SUFFIX first')
else:
    y = np.load(f'{OUT}/parcel_labels_cortical.npy')
    assert np.array_equal(y, np.load(f'{OUT}/parcel_labels_glasser52.npy')), 'subject order differs'
    N = len(y)

    def oof(path):
        """Pooled out-of-fold predictions, same folds for both feature sets."""
        X = np.load(path)
        p, q = np.zeros(N, int), np.zeros(N)
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=123).split(X, y):
            m = LogisticRegression(max_iter=2000).fit(X[tr], y[tr])
            p[te], q[te] = m.predict(X[te]), m.predict_proba(X[te])[:, 1]
        return p, q

    pa, qa = oof(A)
    pb, qb = oof(B)

    # One resample per iteration, applied to BOTH methods, so the shared subject-level
    # variance cancels instead of inflating the interval.
    rng, dA, dU = np.random.default_rng(123), [], []
    for _ in range(2000):
        b = rng.choice(N, N, replace=True)
        if len(np.unique(y[b])) < 2:
            continue
        dA.append(accuracy_score(y[b], pb[b]) - accuracy_score(y[b], pa[b]))
        dU.append(roc_auc_score(y[b], qb[b]) - roc_auc_score(y[b], qa[b]))

    da = accuracy_score(y, pb) - accuracy_score(y, pa)
    du = roc_auc_score(y, qb) - roc_auc_score(y, qa)
    al, ah = np.percentile(dA, [2.5, 97.5])
    ul, uh = np.percentile(dU, [2.5, 97.5])

    print(f'{"210 Brainnetome regions":26s} acc {accuracy_score(y,pa):.3f}   AUC {roc_auc_score(y,qa):.3f}')
    print(f'{"52 Glasser parcels":26s} acc {accuracy_score(y,pb):.3f}   AUC {roc_auc_score(y,qb):.3f}')
    print()
    print('paired difference, 52 minus 210')
    print(f'   accuracy {da:+.3f} [{al:+.3f}, {ah:+.3f}]   '
          f'{"SIGNIFICANT" if al > 0 or ah < 0 else "not significant"}')
    print(f'   AUC      {du:+.3f} [{ul:+.3f}, {uh:+.3f}]   '
          f'{"SIGNIFICANT" if ul > 0 or uh < 0 else "not significant"}')
    print()
    if not (al > 0 or ah < 0) and not (ul > 0 or uh < 0):
        print('Cutting 210 channels to 52 costs nothing measurable. Matching Sid\'s')
        print('parcellation does not handicap BrainOmni, so the model comparison is fair.')

210 Brainnetome regions    acc 0.851   AUC 0.913
52 Glasser parcels         acc 0.870   AUC 0.904

paired difference, 52 minus 210
   accuracy +0.019 [-0.012, +0.050]   not significant
   AUC      -0.009 [-0.033, +0.015]   not significant

Cutting 210 channels to 52 costs nothing measurable. Matching Sid's
parcellation does not handicap BrainOmni, so the model comparison is fair.
